# Lab 3: Ridge Regression and Gradient Boosting With Many Predictors (JPM)

**Owner:** Member 3 · **Marking items covered:** 2. Honest validation, 3. Evaluation against the naive forecast, 4. Modelling quality

**The question this notebook answers:** Does a model with our 50 predictors forecast JPM's next-day log return better than the naive forecast, on days it never saw?

> **How to use this starter (delete this box when you finish).** Steps 1 and 2 already work: run them
> first. Every other step says what to do and which course notebook to copy from. Replace each `# TODO`
> cell with your code, and each *italic* placeholder with your own words about **JPM's** results.
> Your full instructions are in your section of the team guide.

## Step 1: Set up

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tslab import config
from tslab.data import load_market_data
from tslab.team import build_features_jpm, make_dataset_jpm, FEATURE_GROUPS_JPM, ALL_FEATURES_JPM   # OUR features
from tslab.validation import walk_forward_splits, walk_forward_predict, describe_splits
from tslab.metrics import (compare_models, diebold_mariano, dm_verdict, directional_accuracy,
                           directional_accuracy_test, out_of_sample_r2, rmse, mae,
                           simple_backtest, backtest_summary)
from tslab.plots import plot_walk_forward_layout, plot_price_forecasts
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from tslab.features import check_no_lookahead

import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
np.random.seed(config.SEED)
print("Target stock:", config.TARGET_TICKER)

## Step 2: The data and the walk-forward folds

Every notebook in our project uses these lines, so that every model is scored on the same days.

In [ ]:
frame = load_market_data()                                              # one row per JPM trading day
dataset_all = make_dataset_jpm(frame, horizon=1)                        # includes the last row (no target yet)
latest_row = dataset_all.iloc[[-1]]                                     # kept aside for a real forecast
dataset = dataset_all.dropna(subset=["target_return"])
splits = list(walk_forward_splits(len(dataset), config.INITIAL_TRAIN, config.TEST_SIZE))

print("Rows in the modelling table:", len(dataset))                     # expect 2789
print("Walk-forward folds         :", len(splits))                      # expect 29
print("First test day:", dataset.index[splits[0][1][0]].date(), "| last test day:", dataset.index[splits[-1][1][-1]].date())

## Step 3: The table of predictors

Show how many features are in each group of `FEATURE_GROUPS_JPM` (7 groups, 50 features). Copy the idea of Notebook 4, Step 2.

In [ ]:
pd.Series({g: len(cols) for g, cols in FEATURE_GROUPS_JPM.items()}, name="features").to_frame()

## Step 4: Look-ahead bias

Run `check_no_lookahead(frame, build_features_jpm)`, then repeat Notebook 4's `center=True` leak demo on JPM and explain in your words why it leaks.

In [ ]:
assert check_no_lookahead(frame, build_features_jpm)
print("Look-ahead test passed")

# TODO: the center=True / center=False demo (Notebook 4, Step 3.2 and 3.3)

## Step 5: Walk-forward folds

Show the fold picture, and write why the scaler inside `walk_forward_predict` is fitted on the training rows only.

In [ ]:
X, y = dataset[ALL_FEATURES_JPM], dataset["target_return"]
plot_walk_forward_layout(splits, len(dataset))

## Step 6: Tune without touching the test period

Copy Notebook 4, Step 6. Use only `X.iloc[:config.INITIAL_TRAIN]`. **Extend the Ridge grid to 1e7**: on JPM the course grid picks the largest alpha (100,000). Explain what a very large alpha means.

In [ ]:
# TODO: tuning block, Ridge alpha grid, gradient boosting grid -> best_alpha, best_depth, best_rate

## Step 7: Walk-forward test: the main results table

Run Ridge and gradient boosting walk-forward, then `compare_models` (it adds naive) and the DM + hit-rate table (Notebook 4, Step 7). Also print the share of up days in the test period and the share of days each model predicts up.

In [ ]:
predictions = {}
# TODO: predictions["Ridge"] = walk_forward_predict(...)
# TODO: predictions["Gradient boosting"] = walk_forward_predict(...)

## Step 8: Feature group ablation

Copy Notebook 4, Step 8 with `FEATURE_GROUPS_JPM`. Also remove only the TECH features. Explain which group helps; a difference of about 0.001 in R² is not a finding.

In [ ]:
# TODO

## Step 9: Permutation importance

Copy Notebook 4, Step 9, on the test folds only. Top 15 features.

In [ ]:
# TODO

## Step 10: Forecast for the next trading day

Copy Notebook 4, Step 12 with `latest_row[ALL_FEATURES_JPM]`.

In [ ]:
# TODO

## Save

Other members read this file. Keep the file and column names exactly as in the team guide.

In [ ]:
if 'predictions' in globals() and predictions:
    os.makedirs(config.OUTPUT_DIR, exist_ok=True)
    pd.concat(predictions, axis=1).to_csv(config.OUTPUT_DIR / "predictions_ml.csv")
    # json.dump({"best_alpha": best_alpha, "best_depth": best_depth, "best_rate": best_rate},
    #           open(config.OUTPUT_DIR / "ml_best_params.json", "w"))          # TODO: uncomment after Step 6
    print("Saved outputs/predictions_ml.csv")
else:
    print("Nothing saved yet: no models.")

## Conclusion

*TODO (your words, about JPM's numbers). If no model beats the naive forecast, say so plainly: it is a valid finding.*